# 🏥 Hospital OPD Queue & Bed Allocation Optimization
### Discrete-Event Simulation, Machine Learning, and Explainable Reinforcement Learning
**Course:** Advanced Data Science & Operations Analytics  
**Authors:** Data Science Engineering Team  
**Local Time:** 2026-10-18

---

## 📌 1. Project Overview & Clinical Problem Statement
Hospital Outpatient Departments (OPDs) and Emergency Departments worldwide struggle with severe queue congestion, prolonged waiting times, bed shortages, and routine patient starvation.

In this project, we develop an end-to-end data science and operational optimization framework:
1. **Discrete-Event Simulation (SimPy):** Models non-homogeneous Poisson arrivals (peaking 9am–12pm), log-normal consultation times, triage priority streams, and ward/ICU bed allocation.
2. **Classical Machine Learning:**
   - Linear, Multiple, and Polynomial Regression for consultation duration and next-hour arrivals.
   - Logistic Regression for inpatient admission prediction.
   - Decision Tree for triage acuity classification.
   - K-Nearest Neighbors (KNN) for similar-patient consultation estimates.
   - 5-Fold Cross-Validation and comprehensive evaluation tables.
3. **Clustering & Demand Regimes:**
   - K-Means for patient clinical phenotyping (Elbow & Silhouette).
   - Agglomerative Hierarchical Clustering with Ward linkage & Dendrogram for hourly demand regimes (*Normal*, *Peak*, *Surge*).
4. **Recommender Systems:** Content-based, Collaborative Filtering, and Hybrid recommenders for doctor/department and ward/bed assignment (evaluated via Precision@k).
5. **The Four Core Innovations:**
   - **Innovation 1:** Fairness-Aware Aging Priority ($Score = Acuity + \alpha \cdot Wait$)
   - **Innovation 2:** Predict-then-Decide Queue Ordering & Bed Pre-reservation
   - **Innovation 3:** Demand Regime Detection & Adaptive Staffing
   - **Innovation 4:** Explainable RL Policy Distillation into Surrogate Decision Trees
6. **Multi-Seed Evaluation & Hypothesis Testing:** 30 independent seeds, 95% Confidence Intervals, and paired Student's t-tests.
7. **Queuing Theory Validation:** Erlang-C multi-server model and Little's Law ($L = \lambda W$).


In [ ]:
# Setup and Environment Imports
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("Environment setup completed successfully.")


## 📁 2. Data Ingestion & Missing Value Preprocessing
We generate 6 months of synthetic outpatient records (180 days, ~50,000 patients) with documented distributions and realistic missing values.


In [ ]:
from src.preprocessing_eda import load_raw_data, clean_missing_values, compute_descriptive_statistics, compute_correlation_matrices

# Load raw synthetic OPD data
raw_df = load_raw_data("../data/synthetic_opd.csv")
print(f"Loaded raw dataset: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")

# Inspect injected missing values
missing_counts = raw_df.isnull().sum()
print("Missing values in raw data:\n", missing_counts[missing_counts > 0])

# Clean missing values via domain-guided conditional imputation
df, imputation_report = clean_missing_values(raw_df)
print("\nMissing values after domain-guided imputation:\n", df[['systolic_bp', 'temperature', 'satisfaction_score']].isnull().sum())


## 📊 3. Exploratory Data Analysis & Descriptive Statistics
We calculate mean, median, variance, standard deviation, skewness, min, max, and IQR across all numeric clinical variables, and compute Pearson and Spearman correlation matrices.


In [ ]:
# Descriptive Statistics Table
desc_stats = compute_descriptive_statistics(df)
display(desc_stats)

# Correlation Matrix Heatmap
p_corr, s_corr = compute_correlation_matrices(df)

plt.figure(figsize=(10, 7))
sns.heatmap(p_corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", linewidths=0.5)
plt.title("Pearson Correlation Heatmap (Clinical & Operational Features)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 🤖 4. Classical Machine Learning Suite & Model Benchmark Table
We train and evaluate:
1. **Simple, Multiple, and Polynomial Regression** to predict consultation time.
2. **Multiple Linear Regression with Lag Features** to forecast next-hour patient arrivals.
3. **Logistic Regression** to predict patient admission propensity ($0/1$).
4. **Decision Tree Classifier** to classify triage priority (Emergency, Urgent, Routine).
5. **K-Nearest Neighbors (KNN)** to estimate consultation time from similar patients.
6. Evaluate all models with RMSE, MAE, $R^2$, Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix, and 5-Fold Cross-Validation.


In [ ]:
from src.ml_models import MLSuite

ml_suite = MLSuite(data_path="../data/synthetic_opd.csv", seed=42)
benchmark_summary = ml_suite.run_all()

print("=" * 95)
print("CLASSICAL MACHINE LEARNING MODEL BENCHMARK TABLE")
print("=" * 95)
display(benchmark_summary)


## 🔍 5. Clustering & Demand Regime Detection (Innovation 3)
We perform:
1. **K-Means Clustering** for patient phenotyping (evaluating Elbow SSE and Silhouette scores).
2. **Agglomerative Hierarchical Clustering (Ward Linkage)** to segment operational demand into **Normal**, **Peak**, and **Surge** regimes.


In [ ]:
from src.clustering import OPDClusteringAnalysis

clustering_analysis = OPDClusteringAnalysis(data_path="../data/synthetic_opd.csv", seed=42)

# 1. Patient Phenotype Clustering
p_metrics, best_k, p_profiles = clustering_analysis.analyze_patient_clusters(k_range=range(2, 7))
print(f"Optimal Patient Clinical Clusters (k={best_k}):")
display(p_profiles)

# 2. Demand Regime Clustering
r_metrics, r_summary = clustering_analysis.cluster_demand_regimes(n_clusters=3)
print("\nHourly Operational Demand Regimes (Normal, Peak, Surge):")
display(r_summary)


## 💡 6. Clinical Recommender Systems
We compare **Content-Based (TF-IDF + Cosine Similarity)**, **Collaborative Filtering (Matrix Factorization)**, and **Hybrid** recommenders for Doctor/Department and Ward/Bed allocation, evaluated via Precision@k.


In [ ]:
from src.recommender import HospitalRecommenderSuite

rec_suite = HospitalRecommenderSuite(data_path="../data/synthetic_opd.csv", seed=42)

# Test Doctor Recommendations for an Acute Case
query_symptom = "Severe Chest Pain"
print(f"Hybrid Doctor Recommendations for: '{query_symptom}', Age 58, Emergency:")
recs = rec_suite.recommend_hybrid(query_symptom, age=58, age_group="41-65", triage_class="Emergency", top_k=3)
for r in recs:
    print(f"  - {r['doctor_name']} ({r['department']}) -> Score: {r['score']} [Content: {r['content_score']}, Collaborative: {r['collaborative_score']}]")

# Test Bed Recommendation
bed_rec = rec_suite.recommend_ward_and_bed(triage_class="Emergency", oxygen_saturation=89.5, systolic_bp=165.0, predicted_admit_prob=0.82)
print("\nWard/Bed Recommendation:", bed_rec)

# Evaluate Precision@k across models
precision_eval = rec_suite.evaluate_precision_at_k(test_sample_size=300)
display(precision_eval)


## ⚖️ 7. Demographic Fairness & Starvation Sweep (Innovation 1)
We audit waiting times across Age Groups, Gender, and Urban/Rural areas, and sweep the $\alpha$ parameter in Aging Priority:
$$\text{Score}(t) = \text{Acuity} + \alpha \cdot \text{Wait}(t)$$


In [ ]:
from src.fairness import audit_demographic_fairness, sweep_alpha_tradeoff

# Subgroup fairness audit
fairness_dict = audit_demographic_fairness(df)
print("Fairness Audit across Residential Areas (Urban vs Rural):")
display(fairness_dict["area_fairness"])

# Alpha parameter sweep
print("\nSweeping Alpha in Aging Priority:")
sweep_results = sweep_alpha_tradeoff(alphas=[0.0, 0.2, 0.5, 0.8, 1.0, 1.5, 2.0], sim_days=1, seed=42)
display(sweep_results)


## 🧠 8. Explainable RL Policy Distillation (Innovation 4)
We distill the trained Reinforcement Learning policy (PPO) into an interpretable **Surrogate Decision Tree**, quantify its **Fidelity**, and extract feature importances and transparent clinical rules.


In [ ]:
from src.explain_policy import PolicyDistiller

distiller = PolicyDistiller(model_path="../data/trained_models/ppo_hospital_agent.zip", max_depth=5, seed=42)
distill_res = distiller.distill(num_episodes=15)

print(f"Distilled Decision Tree Policy Fidelity: {distill_res['fidelity']*100:.2f}% across {distill_res['total_samples']} states")
print("\nState Feature Importances Driving the RL Policy:")
for f, imp in sorted(distill_res['feature_importances'].items(), key=lambda x: x[1], reverse=True):
    print(f"  - {f:22s}: {imp:.4f}")


## 🔬 9. Multi-Seed Simulation Benchmark & Hypothesis Testing
We evaluate all 6 policies across 30 independent seeds, compute 95% Confidence Intervals using the Student-t distribution, and run paired Student's t-tests ($H_0$: equal mean wait time vs FCFS).


In [ ]:
from src.evaluate import HospitalEvaluator

evaluator = HospitalEvaluator(num_seeds=30)
benchmark_30seeds = evaluator.run_multi_seed_benchmark(sim_days=1)
display(benchmark_30seeds[["Policy", "Mean_Wait_CI", "Emergency_Wait_CI", "Routine_Wait_CI", "Doctor_Utilization", "Starvation_Rate"]])

print("\nPaired Student's t-Tests vs FCFS Baseline:")
ttests_30seeds = evaluator.run_paired_t_tests(baseline_policy="fcfs")
display(ttests_30seeds)


## 📐 10. Queuing Theory (M/M/c) & Little's Law Simulator Validation
We prove the mathematical validity of our discrete-event simulation engine by validating against theoretical Erlang-C formulas and Little's Law ($L = \lambda W$).


In [ ]:
from src.validation import validate_queuing_theory

validation_results = validate_queuing_theory()
print("Erlang-C Analytical Queuing Model vs SimPy Simulation Results:")
display(validation_results)
